# 11 - Wikipedia Enrichment

This notebook enriches the POI importance layer with Wikipedia-based popularity signals.

Goals:
- match important POIs to Wikipedia pages
- collect pageview-based popularity features
- improve landmark-specific importance beyond OSM heuristics

In [158]:
import time
import requests
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Load enriched POIs
We start from the output of `10_poi_enrichment.ipynb`.

In [159]:
pois = pd.read_csv("../data/processed/poi_enriched.csv")
pois.head()

,poi_id,name,name_en,category,category_clean,lat,lon,distance_to_center_km,nearby_count_500m,cluster_id,category_score,landmark_name_score,centrality_score,density_score,importance_score,is_park,is_historic,is_museum,is_attraction,is_religious
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,historic:archaeological_site,historic,41.007586,28.975554,0.248384,64,12,0.82,0.55,0.997367,0.941176,0.863004,0,1,0,0,0
1,1075801479,Antiochos Sarayı'nın Kalıntıları,NaN,historic:archaeological_site,historic,41.007289,28.975329,0.276907,63,12,0.82,0.55,0.997021,0.926471,0.859224,0,1,0,0,0
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic:castle,historic,41.006397,28.974808,0.361956,63,12,0.82,0.55,0.995990,0.926471,0.858915,0,1,0,0,0
3,311681431,Sağlık Müzesi,NaN,tourism:museum,museum,41.008314,28.975290,0.261290,66,12,0.85,0.35,0.997210,0.970588,0.849310,0,0,1,0,0
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,tourism:museum,museum,41.004295,28.977433,0.441766,55,12,0.85,0.59,0.995023,0.808824,0.844213,0,0,1,0,0


In [160]:
pois["match_name"] = pois["name_en"].fillna(pois["name"])
pois[["name", "name_en", "match_name"]].head(10)

,name,name_en,match_name
0,Lausos Sarayı'nın Kalıntıları,NaN,Lausos Sarayı'nın Kalıntıları
1,Antiochos Sarayı'nın Kalıntıları,NaN,Antiochos Sarayı'nın Kalıntıları
2,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,Ibrahim Pasha Palace
3,Sağlık Müzesi,NaN,Sağlık Müzesi
4,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,Great Palace Mosaic Museum
5,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,Tomb of Sultan Ahmed I
6,Ayasofya Tarih ve Deneyim Müzesi,Hagia Sophia History and Experience Museum,Hagia Sophia History and Experience Museum
7,Türk ve İslam Eserleri Müzesi,Turkish and Islamic Arts Museum,Turkish and Islamic Arts Museum
8,Halı Müzesi,Carpet Museum,Carpet Museum
9,Keçecizâde Fuad Paşa Türbesi,Turbe of Keçecizâde Fuad Pasha,Turbe of Keçecizâde Fuad Pasha


In [161]:
pois["name_en"].notna().sum(), len(pois)

(np.int64(307), 2761)

In [162]:
print("Shape:", pois.shape)
print(pois.columns.tolist())

Shape: (2761, 21)
['poi_id', 'name', 'name_en', 'category', 'category_clean', 'lat', 'lon', 'distance_to_center_km', 'nearby_count_500m', 'cluster_id', 'category_score', 'landmark_name_score', 'centrality_score', 'density_score', 'importance_score', 'is_park', 'is_historic', 'is_museum', 'is_attraction', 'is_religious', 'match_name']


## Start with the most important POIs
To keep the first Wikipedia pass manageable, we begin with the top-ranked POIs.

In [163]:
top_pois = pois.sort_values("importance_score", ascending=False).head(50).copy()
top_pois.head(20)

,poi_id,name,name_en,category,category_clean,lat,lon,distance_to_center_km,nearby_count_500m,cluster_id,category_score,landmark_name_score,centrality_score,density_score,importance_score,is_park,is_historic,is_museum,is_attraction,is_religious,match_name
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,historic:archaeological_site,historic,41.007586,28.975554,0.248384,64,12,0.82,0.55,0.997367,0.941176,0.863004,0,1,0,0,0,Lausos Sarayı'nın Kalıntıları
1,1075801479,Antiochos Sarayı'nın Kalıntıları,NaN,historic:archaeological_site,historic,41.007289,28.975329,0.276907,63,12,0.82,0.55,0.997021,0.926471,0.859224,0,1,0,0,0,Antiochos Sarayı'nın Kalıntıları
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic:castle,historic,41.006397,28.974808,0.361956,63,12,0.82,0.55,0.995990,0.926471,0.858915,0,1,0,0,0,Ibrahim Pasha Palace
3,311681431,Sağlık Müzesi,NaN,tourism:museum,museum,41.008314,28.975290,0.261290,66,12,0.85,0.35,0.997210,0.970588,0.849310,0,0,1,0,0,Sağlık Müzesi
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,tourism:museum,museum,41.004295,28.977433,0.441766,55,12,0.85,0.59,0.995023,0.808824,0.844213,0,0,1,0,0,Great Palace Mosaic Museum
5,103953125,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,tourism:attraction,attraction,41.006789,28.976985,0.196703,68,12,0.78,0.35,0.997993,1.000000,0.835898,0,0,0,1,1,Tomb of Sultan Ahmed I
6,11867279469,Ayasofya Tarih ve Deneyim Müzesi,Hagia Sophia History and Experience Museum,tourism:museum,museum,41.006455,28.975367,0.320059,62,12,0.85,0.35,0.996498,0.911765,0.834391,0,0,1,0,0,Hagia Sophia History and Experience Museum
7,5113500256,Türk ve İslam Eserleri Müzesi,Turkish and Islamic Arts Museum,tourism:museum,museum,41.006280,28.974915,0.362063,62,12,0.85,0.35,0.995989,0.911765,0.834238,0,0,1,0,0,Turkish and Islamic Arts Museum
8,3373094254,Halı Müzesi,Carpet Museum,tourism:museum,museum,41.005682,28.978669,0.280933,60,12,0.85,0.35,0.996972,0.882353,0.827180,0,0,1,0,0,Carpet Museum
9,527580309,Keçecizâde Fuad Paşa Türbesi,Turbe of Keçecizâde Fuad Pasha,historic:tomb,historic,41.006561,28.972843,0.500662,61,12,0.82,0.35,0.994309,0.897059,0.821057,0,1,0,0,1,Turbe of Keçecizâde Fuad Pasha


## Helper functions for Wikipedia matching
We use the MediaWiki search API first.
Later we can improve this with geosearch and better matching logic.

In [164]:
WIKIPEDIA_API_URL = "https://en.wikipedia.org/w/api.php"
USER_AGENT = "tourism-crowd-forecasting-research/1.0"


def search_wikipedia(query, limit=5, max_retries=5):
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": limit,
    }

    for attempt in range(max_retries):
        response = requests.get(
            WIKIPEDIA_API_URL,
            params=params,
            headers={"User-Agent": USER_AGENT},
            timeout=20,
        )

        if response.status_code == 200:
            return response.json()

        if response.status_code == 429:
            wait_time = 2 ** attempt
            print(f"Rate limited for '{query}', waiting {wait_time}s...")
            time.sleep(wait_time)
            continue

        response.raise_for_status()

    raise Exception(f"Wikipedia search failed after retries for query: {query}")


In [165]:
import re
import unicodedata

In [166]:
STOPWORDS = {
    "the", "of", "and", "in", "at", "for",
    "museum", "museu", "müzesi", "muzesi",
    "mosque", "camii", "cami",
    "palace", "saray", "sarayı", "sarayi",
    "tower", "kule",
    "church", "kilise",
    "park", "tomb", "turbe", "türbe",
    "square", "meydani", "meydanı",
    "complex", "list", "monuments", "istanbul"
}


ALIASES = {
    "ayasofya": ["hagia", "sophia", "ayasofya"],
    "topkapi": ["topkapi", "topkapı"],
    "topkapı": ["topkapi", "topkapı"],
    "dolmabahce": ["dolmabahce", "dolmabahçe"],
    "dolmabahçe": ["dolmabahce", "dolmabahçe"],
    "sultanahmet": ["sultanahmet", "sultan", "ahmed"],
    "sarnici": ["cistern", "sarnic", "sarnıcı", "sarnici"],
    "sarnıcı": ["cistern", "sarnic", "sarnıcı", "sarnici"],
    "galata": ["galata"],
    "kiz": ["kiz", "kız", "maiden"],
    "kız": ["kiz", "kız", "maiden"],
}

In [167]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).casefold()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [168]:
def meaningful_tokens(text):
    text = normalize_text(text)
    tokens = text.split()

    expanded = []
    for token in tokens:
        if token in STOPWORDS:
            continue
        expanded.append(token)
        if token in ALIASES:
            expanded.extend(ALIASES[token])

    expanded = [tok for tok in expanded if tok not in STOPWORDS and len(tok) >= 3]
    return sorted(set(expanded))

In [169]:
def token_overlap_score(poi_name, wiki_title):
    poi_tokens = set(meaningful_tokens(poi_name))
    wiki_tokens = set(meaningful_tokens(wiki_title))

    if not poi_tokens or not wiki_tokens:
        return 0.0

    overlap = poi_tokens.intersection(wiki_tokens)
    return len(overlap) / len(poi_tokens)

In [170]:
QUERY_REPLACEMENTS = {
    "ayasofya": ["hagia sophia", "ayasofya"],
    "topkapı": ["topkapi", "topkapi palace", "topkapı"],
    "topkapi": ["topkapi", "topkapi palace", "topkapı"],
    "dolmabahçe": ["dolmabahce", "dolmabahce palace", "dolmabahçe"],
    "dolmabahce": ["dolmabahce", "dolmabahce palace", "dolmabahçe"],
    "yerebatan": ["basilica cistern", "yerebatan"],
    "sarnıcı": ["cistern"],
    "sarnici": ["cistern"],
    "sultanahmet": ["blue mosque", "sultan ahmed", "sultanahmet"],
    "kapalıçarşı": ["grand bazaar", "kapalicarsi", "kapalıçarşı"],
    "kapalicarsi": ["grand bazaar", "kapalicarsi", "kapalıçarşı"],
    "mısır çarşısı": ["spice bazaar", "egyptian bazaar", "misir carsisi"],
    "misir carsisi": ["spice bazaar", "egyptian bazaar", "misir carsisi"],
    "kız kulesi": ["maiden's tower", "kiz kulesi", "maiden tower"],
    "kiz kulesi": ["maiden's tower", "kiz kulesi", "maiden tower"],
    "galata": ["galata tower", "galata"],
}

In [171]:
def generate_query_candidates(name):
    normalized = normalize_text(name)
    candidates = [name]

    for key, replacements in QUERY_REPLACEMENTS.items():
        if key in normalized:
            candidates.extend(replacements)

    candidates.append(f"{name} Istanbul")

    seen = set()
    unique_candidates = []
    for candidate in candidates:
        candidate = candidate.strip()
        if candidate and candidate not in seen:
            seen.add(candidate)
            unique_candidates.append(candidate)

    return unique_candidates

In [172]:
def get_best_wikipedia_match(name, min_overlap=0.50):
    try:
        banned_title_tokens = {
            "list", "disambiguation", "ankara",
            "mustafa", "kemal", "atatürk", "ataturk",
            "film", "album", "song", "novel", "tv", "series"
        }

        place_hint_map = {
            "saray": {"palace"},
            "sarayi": {"palace"},
            "sarayı": {"palace"},
            "cami": {"mosque"},
            "camii": {"mosque"},
            "mosque": {"mosque"},
            "müze": {"museum"},
            "muze": {"museum"},
            "museum": {"museum"},
            "sarnic": {"cistern"},
            "sarnici": {"cistern"},
            "sarnıcı": {"cistern"},
            "kule": {"tower"},
            "tower": {"tower"},
            "meydan": {"square"},
            "meydani": {"square"},
            "meydanı": {"square"},
            "kilise": {"church"},
            "church": {"church"},
            "tomb": {"tomb", "mausoleum"},
            "turbe": {"tomb", "mausoleum"},
            "türbe": {"tomb", "mausoleum"},
            "mausoleum": {"tomb", "mausoleum"},
            "column": {"column", "obelisk"},
            "sutun": {"column", "obelisk"},
            "sütun": {"column", "obelisk"},
            "dikilitas": {"obelisk", "column"},
            "dikilitaş": {"obelisk", "column"},
        }

        person_like_words = {
            "born", "died", "statesman", "poet", "writer", "politician",
            "actor", "actress", "singer", "composer", "pasha", "sultan"
        }

        all_scored = []
        query_tokens = set(meaningful_tokens(name))
        normalized_name = normalize_text(name)
        query_candidates = generate_query_candidates(name)

        expected_place_tokens = set()
        for hint, mapped_tokens in place_hint_map.items():
            if hint in normalized_name:
                expected_place_tokens.update(mapped_tokens)

        for query in query_candidates:
            result = search_wikipedia(query, limit=5)
            matches = result.get("query", {}).get("search", [])

            for item in matches:
                title = item.get("title", "")
                snippet = item.get("snippet", "")
                title_tokens = set(meaningful_tokens(title))
                overlap = token_overlap_score(name, title)

                normalized_title = normalize_text(title)
                normalized_snippet = normalize_text(snippet)

                if any(tok in banned_title_tokens for tok in title_tokens):
                    continue

                if len(query_tokens.intersection(title_tokens)) == 0 and overlap < min_overlap:
                    continue

                score = overlap

                # Prefer place-like titles when POI name suggests a place
                if expected_place_tokens and any(tok in title_tokens for tok in expected_place_tokens):
                    score += 0.30

                if expected_place_tokens and not any(tok in title_tokens for tok in expected_place_tokens):
                    score -= 0.20

                # Penalize person-like pages when we expect a place
                if expected_place_tokens:
                    if any(word in normalized_title for word in person_like_words):
                        score -= 0.35
                    if any(word in normalized_snippet for word in person_like_words):
                        score -= 0.20

                suspicious_words = ["film", "album", "song", "novel", "actor", "politician"]
                if any(word in normalized_title for word in suspicious_words):
                    score -= 0.50
                if any(word in normalized_snippet for word in suspicious_words):
                    score -= 0.30

                all_scored.append((score, overlap, query, item))

            time.sleep(0.5)

        if not all_scored:
            return {
                "wiki_matched": False,
                "wiki_title": None,
                "wiki_pageid": None,
                "wiki_snippet": None,
                "wiki_overlap_score": 0.0,
                "wiki_query_used": None,
            }

        all_scored.sort(key=lambda x: x[0], reverse=True)
        best_score, best_overlap, best_query, best_item = all_scored[0]

        if best_overlap < min_overlap:
            return {
                "wiki_matched": False,
                "wiki_title": None,
                "wiki_pageid": None,
                "wiki_snippet": None,
                "wiki_overlap_score": best_overlap,
                "wiki_query_used": best_query,
            }

        return {
            "wiki_matched": True,
            "wiki_title": best_item.get("title"),
            "wiki_pageid": best_item.get("pageid"),
            "wiki_snippet": best_item.get("snippet"),
            "wiki_overlap_score": best_overlap,
            "wiki_query_used": best_query,
        }

    except Exception as e:
        return {
            "wiki_matched": False,
            "wiki_title": None,
            "wiki_pageid": None,
            "wiki_snippet": f"ERROR: {e}",
            "wiki_overlap_score": 0.0,
            "wiki_query_used": None,
        }


## Quick manual search test
We test a few well-known POIs first.

In [173]:
test_queries = [
    "Ayasofya",
    "Topkapi Palace Istanbul",
    "Dolmabahce Palace",
    "Galata Tower",
    "Kiz Kulesi Istanbul"
]

for query in test_queries:
    print(f"\n===== {query} =====")
    try:
        result = search_wikipedia(query, limit=3)
        for item in result["query"]["search"]:
            print(item["title"])
    except Exception as e:
        print("ERROR:", e)



===== Ayasofya =====
Hagia Sophia
Ayasofya Mosque
Hagia Sophia, Trabzon

===== Topkapi Palace Istanbul =====
Topkapı Palace
Historic Areas of Istanbul
Dolmabahçe Palace

===== Dolmabahce Palace =====
Dolmabahçe Palace
Beşiktaş Stadium
Dolmabahçe Mosque

===== Galata Tower =====
Galata Tower
Galata
Golden Horn

===== Kiz Kulesi Istanbul =====
Maiden's Tower
The Amazing Race 7
Anadoluhisarı


## Apply matching to top POIs
We use the POI name directly as the first search query.

In [174]:
wiki_matches = []

for _, row in top_pois.iterrows():
    query = row["match_name"]
    match = get_best_wikipedia_match(query)

    wiki_matches.append({
        "poi_id": row["poi_id"],
        "name": row["name"],
        "name_en": row["name_en"],
        "match_name": row["match_name"],
        "importance_score": row["importance_score"],
        "wiki_matched": match["wiki_matched"],
        "wiki_title": match["wiki_title"],
        "wiki_pageid": match["wiki_pageid"],
        "wiki_snippet": match["wiki_snippet"],
        "wiki_overlap_score": match["wiki_overlap_score"],
        "wiki_query_used": match["wiki_query_used"],
    })

    time.sleep(1.0)

wiki_matches_df = pd.DataFrame(wiki_matches)
wiki_matches_df.head(20)


,poi_id,name,name_en,match_name,importance_score,wiki_matched,wiki_title,wiki_pageid,wiki_snippet,wiki_overlap_score,wiki_query_used
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,Lausos Sarayı'nın Kalıntıları,0.863004,False,NaN,NaN,NaN,0.0,NaN
1,1075801479,Antiochos Sarayı'nın Kalıntıları,NaN,Antiochos Sarayı'nın Kalıntıları,0.859224,False,NaN,NaN,NaN,0.0,NaN
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,Ibrahim Pasha Palace,0.858915,True,Pargalı Ibrahim Pasha,2794458.0,"Pargalı <span class=""searchmatch"">Ibrahim</spa...",1.0,Ibrahim Pasha Palace
3,311681431,Sağlık Müzesi,NaN,Sağlık Müzesi,0.849310,False,NaN,NaN,NaN,0.0,NaN
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,Great Palace Mosaic Museum,0.844213,True,Great Palace Mosaic Museum,4347571.0,"The <span class=""searchmatch"">Great</span> <sp...",1.0,Great Palace Mosaic Museum
5,103953125,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,Tomb of Sultan Ahmed I,0.835898,True,Ahmed III,1529.0,"<span class=""searchmatch"">Ahmed</span> III (Ot...",0.5,Tomb of Sultan Ahmed I
6,11867279469,Ayasofya Tarih ve Deneyim Müzesi,Hagia Sophia History and Experience Museum,Hagia Sophia History and Experience Museum,0.834391,True,Hagia Sophia,42764.0,"<span class=""searchmatch"">Hagia</span> <span c...",0.5,Hagia Sophia History and Experience Museum
7,5113500256,Türk ve İslam Eserleri Müzesi,Turkish and Islamic Arts Museum,Turkish and Islamic Arts Museum,0.834238,True,Turkish and Islamic Arts Museum,8174199.0,"The <span class=""searchmatch"">Turkish</span> <...",1.0,Turkish and Islamic Arts Museum
8,3373094254,Halı Müzesi,Carpet Museum,Carpet Museum,0.827180,True,Museum of Carpet,62201527.0,"The <span class=""searchmatch"">Museum</span> of...",1.0,Carpet Museum
9,527580309,Keçecizâde Fuad Paşa Türbesi,Turbe of Keçecizâde Fuad Pasha,Turbe of Keçecizâde Fuad Pasha,0.821057,False,NaN,NaN,NaN,0.0,NaN


In [175]:
wiki_matches_df["wiki_matched"].value_counts(dropna=False)

wiki_matched
False    25
True     25
Name: count, dtype: int64

## Inspect top matches
This is where we manually see whether the raw matching is sensible.

In [176]:
wiki_matches_df[[
    "name",
    "name_en",
    "match_name",
    "wiki_matched",
    "wiki_title",
    "wiki_overlap_score",
    "wiki_query_used"
]].head(50)

,name,name_en,match_name,wiki_matched,wiki_title,wiki_overlap_score,wiki_query_used
0,Lausos Sarayı'nın Kalıntıları,NaN,Lausos Sarayı'nın Kalıntıları,False,NaN,0.000000,NaN
1,Antiochos Sarayı'nın Kalıntıları,NaN,Antiochos Sarayı'nın Kalıntıları,False,NaN,0.000000,NaN
2,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,Ibrahim Pasha Palace,True,Pargalı Ibrahim Pasha,1.000000,Ibrahim Pasha Palace
3,Sağlık Müzesi,NaN,Sağlık Müzesi,False,NaN,0.000000,NaN
4,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,Great Palace Mosaic Museum,True,Great Palace Mosaic Museum,1.000000,Great Palace Mosaic Museum
5,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,Tomb of Sultan Ahmed I,True,Ahmed III,0.500000,Tomb of Sultan Ahmed I
6,Ayasofya Tarih ve Deneyim Müzesi,Hagia Sophia History and Experience Museum,Hagia Sophia History and Experience Museum,True,Hagia Sophia,0.500000,Hagia Sophia History and Experience Museum
7,Türk ve İslam Eserleri Müzesi,Turkish and Islamic Arts Museum,Turkish and Islamic Arts Museum,True,Turkish and Islamic Arts Museum,1.000000,Turkish and Islamic Arts Museum
8,Halı Müzesi,Carpet Museum,Carpet Museum,True,Museum of Carpet,1.000000,Carpet Museum
9,Keçecizâde Fuad Paşa Türbesi,Turbe of Keçecizâde Fuad Pasha,Turbe of Keçecizâde Fuad Pasha,False,NaN,0.000000,NaN


In [177]:
wiki_matches_df[wiki_matches_df["wiki_matched"] == True][[
    "name",
    "name_en",
    "match_name",
    "wiki_title",
    "wiki_overlap_score",
    "wiki_query_used"
]].sort_values(
    ["wiki_overlap_score", "name"],
    ascending=[False, True]
).head(50)

,name,name_en,match_name,wiki_title,wiki_overlap_score,wiki_query_used
31,Alman Çeşmesi,German Fountain,German Fountain,German Fountain,1.0,German Fountain
23,Aya İrini Kilisesi,Hagia Irene,Hagia Irene,Hagia Irene,1.0,Hagia Irene
16,Binbirdirek Sarnıcı,Cistern of Philoxenos,Cistern of Philoxenos,Cistern of Philoxenos,1.0,Cistern of Philoxenos
4,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,Great Palace Mosaic Museum,Great Palace Mosaic Museum,1.0,Great Palace Mosaic Museum
32,Eski Şark Eserleri Müzesi,Museum Of Ancient Orient,Museum Of Ancient Orient,Museum of the Ancient Orient,1.0,Museum Of Ancient Orient
8,Halı Müzesi,Carpet Museum,Carpet Museum,Museum of Carpet,1.0,Carpet Museum
19,Halı Müzesi,Carpet Museum,Carpet Museum,Museum of Carpet,1.0,Carpet Museum
39,Katip Sinan Camii,NaN,Katip Sinan Camii,Katip Sinan Qelebi Mosque,1.0,Katip Sinan Camii
24,Milyon Taşı,Milion,Milion,Milion,1.0,Milion
13,Sultanahmet Camii,Blue Mosque,Blue Mosque,Blue Mosque,1.0,Blue Mosque


## Keep only matched POIs for now

In [178]:
matched_only = wiki_matches_df[wiki_matches_df["wiki_matched"] == True].copy()
matched_only.head(30)

,poi_id,name,name_en,match_name,importance_score,wiki_matched,wiki_title,wiki_pageid,wiki_snippet,wiki_overlap_score,wiki_query_used
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,Ibrahim Pasha Palace,0.858915,True,Pargalı Ibrahim Pasha,2794458.0,"Pargalı <span class=""searchmatch"">Ibrahim</spa...",1.0,Ibrahim Pasha Palace
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,Great Palace Mosaic Museum,0.844213,True,Great Palace Mosaic Museum,4347571.0,"The <span class=""searchmatch"">Great</span> <sp...",1.0,Great Palace Mosaic Museum
5,103953125,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,Tomb of Sultan Ahmed I,0.835898,True,Ahmed III,1529.0,"<span class=""searchmatch"">Ahmed</span> III (Ot...",0.5,Tomb of Sultan Ahmed I
6,11867279469,Ayasofya Tarih ve Deneyim Müzesi,Hagia Sophia History and Experience Museum,Hagia Sophia History and Experience Museum,0.834391,True,Hagia Sophia,42764.0,"<span class=""searchmatch"">Hagia</span> <span c...",0.5,Hagia Sophia History and Experience Museum
7,5113500256,Türk ve İslam Eserleri Müzesi,Turkish and Islamic Arts Museum,Turkish and Islamic Arts Museum,0.834238,True,Turkish and Islamic Arts Museum,8174199.0,"The <span class=""searchmatch"">Turkish</span> <...",1.0,Turkish and Islamic Arts Museum
8,3373094254,Halı Müzesi,Carpet Museum,Carpet Museum,0.827180,True,Museum of Carpet,62201527.0,"The <span class=""searchmatch"">Museum</span> of...",1.0,Carpet Museum
13,18055570,Sultanahmet Camii,Blue Mosque,Blue Mosque,0.807981,True,Blue Mosque,343731.0,"<span class=""searchmatch"">Blue</span> <span cl...",1.0,Blue Mosque
14,132277807,Şehzadeler Türbesi,Mausoleum of Princes,Mausoleum of Princes,0.807860,True,Mausoleum of Safavid Princes,41391331.0,"The <span class=""searchmatch"">Mausoleum</span>...",1.0,Mausoleum of Princes
16,2472730735,Binbirdirek Sarnıcı,Cistern of Philoxenos,Cistern of Philoxenos,0.804169,True,Cistern of Philoxenos,16621019.0,"The <span class=""searchmatch"">Cistern</span> <...",1.0,Cistern of Philoxenos
17,109836395,Sultan III. Murad Türbesi,Mausoleum of Sultan Murad III,Mausoleum of Sultan Murad III,0.804154,True,Murad III,19989.0,"<span class=""searchmatch"">Murad</span> <span c...",0.5,Mausoleum of Sultan Murad III


In [179]:
wiki_review_candidates = wiki_matches_df[wiki_matches_df["wiki_matched"] == True][[
    "poi_id",
    "name",
    "name_en",
    "match_name",
    "wiki_title",
    "wiki_overlap_score",
    "wiki_query_used"
]].copy()

wiki_review_candidates["wiki_match_type"] = ""
wiki_review_candidates["wiki_match_quality"] = ""
wiki_review_candidates["review_notes"] = ""

wiki_review_candidates.to_csv(
    "../data/processed/wiki_review_candidates_top50.csv",
    index=False
)

print("Saved: ../data/processed/wiki_review_candidates_top50.csv")
print("Rows:", len(wiki_review_candidates))

wiki_review_candidates.head(30)

Saved: ../data/processed/wiki_review_candidates_top50.csv
Rows: 25


,poi_id,name,name_en,match_name,wiki_title,wiki_overlap_score,wiki_query_used,wiki_match_type,wiki_match_quality,review_notes
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,Ibrahim Pasha Palace,Pargalı Ibrahim Pasha,1.0,Ibrahim Pasha Palace,,,
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,Great Palace Mosaic Museum,Great Palace Mosaic Museum,1.0,Great Palace Mosaic Museum,,,
5,103953125,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,Tomb of Sultan Ahmed I,Ahmed III,0.5,Tomb of Sultan Ahmed I,,,
6,11867279469,Ayasofya Tarih ve Deneyim Müzesi,Hagia Sophia History and Experience Museum,Hagia Sophia History and Experience Museum,Hagia Sophia,0.5,Hagia Sophia History and Experience Museum,,,
7,5113500256,Türk ve İslam Eserleri Müzesi,Turkish and Islamic Arts Museum,Turkish and Islamic Arts Museum,Turkish and Islamic Arts Museum,1.0,Turkish and Islamic Arts Museum,,,
8,3373094254,Halı Müzesi,Carpet Museum,Carpet Museum,Museum of Carpet,1.0,Carpet Museum,,,
13,18055570,Sultanahmet Camii,Blue Mosque,Blue Mosque,Blue Mosque,1.0,Blue Mosque,,,
14,132277807,Şehzadeler Türbesi,Mausoleum of Princes,Mausoleum of Princes,Mausoleum of Safavid Princes,1.0,Mausoleum of Princes,,,
16,2472730735,Binbirdirek Sarnıcı,Cistern of Philoxenos,Cistern of Philoxenos,Cistern of Philoxenos,1.0,Cistern of Philoxenos,,,
17,109836395,Sultan III. Murad Türbesi,Mausoleum of Sultan Murad III,Mausoleum of Sultan Murad III,Murad III,0.5,Mausoleum of Sultan Murad III,,,


## Wikipedia pageviews
Now we fetch pageview data for matched page titles.

In [180]:
def get_wikipedia_pageviews(title, start="20210101", end="20241231"):
    safe_title = title.replace(" ", "_")
    url = (
        "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
        f"en.wikipedia.org/all-access/user/{safe_title}/daily/{start}/{end}"
    )

    response = requests.get(
        url,
        headers={"User-Agent": USER_AGENT},
        timeout=20,
    )

    if response.status_code != 200:
        return None

    data = response.json()
    return data.get("items", [])

## Test pageviews on a few titles

In [181]:
sample_titles = matched_only["wiki_title"].dropna().head(5).tolist()

for title in sample_titles:
    print(f"\n===== {title} =====")
    pageviews = get_wikipedia_pageviews(title)
    if pageviews is None:
        print("No pageview data")
    else:
        print("Days returned:", len(pageviews))
        print("First record:", pageviews[0] if len(pageviews) > 0 else None)


===== Pargalı Ibrahim Pasha =====
Days returned: 1461
First record: {'project': 'en.wikipedia', 'article': 'Pargalı_Ibrahim_Pasha', 'granularity': 'daily', 'timestamp': '2021010100', 'access': 'all-access', 'agent': 'user', 'views': 432}

===== Great Palace Mosaic Museum =====
Days returned: 1461
First record: {'project': 'en.wikipedia', 'article': 'Great_Palace_Mosaic_Museum', 'granularity': 'daily', 'timestamp': '2021010100', 'access': 'all-access', 'agent': 'user', 'views': 16}

===== Ahmed III =====
Days returned: 1461
First record: {'project': 'en.wikipedia', 'article': 'Ahmed_III', 'granularity': 'daily', 'timestamp': '2021010100', 'access': 'all-access', 'agent': 'user', 'views': 404}

===== Hagia Sophia =====
Days returned: 1461
First record: {'project': 'en.wikipedia', 'article': 'Hagia_Sophia', 'granularity': 'daily', 'timestamp': '2021010100', 'access': 'all-access', 'agent': 'user', 'views': 3484}

===== Turkish and Islamic Arts Museum =====
Days returned: 1461
First recor

## Aggregate pageview statistics
For now we compute:
- total pageviews
- average daily pageviews
- max daily pageviews

In [182]:
pageview_rows = []

for i, row in matched_only.iterrows():
    title = row["wiki_title"]
    pv_data = get_wikipedia_pageviews(title)
    
    if pv_data is None or len(pv_data) == 0:
        pageview_rows.append({
            "poi_id": row["poi_id"],
            "wiki_title": title,
            "wiki_pageviews_total": np.nan,
            "wiki_pageviews_avg": np.nan,
            "wiki_pageviews_max": np.nan,
        })
    else:
        views = [item["views"] for item in pv_data]
        pageview_rows.append({
            "poi_id": row["poi_id"],
            "wiki_title": title,
            "wiki_pageviews_total": np.sum(views),
            "wiki_pageviews_avg": np.mean(views),
            "wiki_pageviews_max": np.max(views),
        })
    
    time.sleep(1.0)

pageviews_df = pd.DataFrame(pageview_rows)
pageviews_df.head()

,poi_id,wiki_title,wiki_pageviews_total,wiki_pageviews_avg,wiki_pageviews_max
0,8120955,Pargalı Ibrahim Pasha,832040,569.500342,1629
1,1153966162,Great Palace Mosaic Museum,35723,24.451061,161
2,103953125,Ahmed III,561542,384.354552,1050
3,11867279469,Hagia Sophia,6419823,4394.129363,23001
4,5113500256,Turkish and Islamic Arts Museum,51912,35.531828,79


## Merge Wikipedia match and pageviews

In [183]:
def minmax_scale(series):
    min_val = series.min()
    max_val = series.max()
    if pd.isna(min_val) or pd.isna(max_val) or min_val == max_val:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - min_val) / (max_val - min_val)


wiki_enriched = wiki_matches_df.merge(
    pageviews_df,
    on=["poi_id", "wiki_title"],
    how="left",
)

wiki_enriched["wiki_popularity_score"] = minmax_scale(
    wiki_enriched["wiki_pageviews_avg"].fillna(0)
)

wiki_enriched[[
    "name",
    "wiki_title",
    "wiki_pageviews_avg",
    "wiki_popularity_score"
]].sort_values("wiki_popularity_score", ascending=False).head(20)

,name,wiki_title,wiki_pageviews_avg,wiki_popularity_score
6,Ayasofya Tarih ve Deneyim Müzesi,Hagia Sophia,4394.129363,1.000000
17,Sultan III. Murad Türbesi,Murad III,1007.997947,0.229397
21,Sultan III. Mehmet Türbesi,Mehmed III,976.321013,0.222188
2,İbrahim Paşa Sarayı,Pargalı Ibrahim Pasha,569.500342,0.129605
5,Sultan Ahmet Türbesi,Ahmed III,384.354552,0.087470
23,Aya İrini Kilisesi,Hagia Irene,165.818617,0.037736
35,Theodosius Dikilitaşı,Obelisk of Theodosius,142.479124,0.032425
46,Yılanlı Sütun,Serpent Column,103.564682,0.023569
24,Milyon Taşı,Milion,45.934292,0.010454
42,SARNIÇ CISTERN,Cistern of Pulcheria,41.931034,0.009543


## Merge back into the top POI table

In [184]:
top_pois_wiki = top_pois.merge(
    wiki_enriched[[
        "poi_id",
        "wiki_matched",
        "wiki_title",
        "wiki_pageid",
        "wiki_pageviews_total",
        "wiki_pageviews_avg",
        "wiki_pageviews_max",
        "wiki_popularity_score"
    ]],
    on="poi_id",
    how="left"
)

top_pois_wiki.head(20)

,poi_id,name,name_en,category,category_clean,lat,lon,distance_to_center_km,nearby_count_500m,cluster_id,category_score,landmark_name_score,centrality_score,density_score,importance_score,is_park,is_historic,is_museum,is_attraction,is_religious,match_name,wiki_matched,wiki_title,wiki_pageid,wiki_pageviews_total,wiki_pageviews_avg,wiki_pageviews_max,wiki_popularity_score
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,historic:archaeological_site,historic,41.007586,28.975554,0.248384,64,12,0.82,0.55,0.997367,0.941176,0.863004,0,1,0,0,0,Lausos Sarayı'nın Kalıntıları,False,NaN,NaN,NaN,NaN,NaN,0.000000
1,1075801479,Antiochos Sarayı'nın Kalıntıları,NaN,historic:archaeological_site,historic,41.007289,28.975329,0.276907,63,12,0.82,0.55,0.997021,0.926471,0.859224,0,1,0,0,0,Antiochos Sarayı'nın Kalıntıları,False,NaN,NaN,NaN,NaN,NaN,0.000000
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic:castle,historic,41.006397,28.974808,0.361956,63,12,0.82,0.55,0.995990,0.926471,0.858915,0,1,0,0,0,Ibrahim Pasha Palace,True,Pargalı Ibrahim Pasha,2794458.0,832040.0,569.500342,1629.0,0.129605
3,311681431,Sağlık Müzesi,NaN,tourism:museum,museum,41.008314,28.975290,0.261290,66,12,0.85,0.35,0.997210,0.970588,0.849310,0,0,1,0,0,Sağlık Müzesi,False,NaN,NaN,NaN,NaN,NaN,0.000000
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,tourism:museum,museum,41.004295,28.977433,0.441766,55,12,0.85,0.59,0.995023,0.808824,0.844213,0,0,1,0,0,Great Palace Mosaic Museum,True,Great Palace Mosaic Museum,4347571.0,35723.0,24.451061,161.0,0.005564
5,103953125,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,tourism:attraction,attraction,41.006789,28.976985,0.196703,68,12,0.78,0.35,0.997993,1.000000,0.835898,0,0,0,1,1,Tomb of Sultan Ahmed I,True,Ahmed III,1529.0,561542.0,384.354552,1050.0,0.087470
6,11867279469,Ayasofya Tarih ve Deneyim Müzesi,Hagia Sophia History and Experience Museum,tourism:museum,museum,41.006455,28.975367,0.320059,62,12,0.85,0.35,0.996498,0.911765,0.834391,0,0,1,0,0,Hagia Sophia History and Experience Museum,True,Hagia Sophia,42764.0,6419823.0,4394.129363,23001.0,1.000000
7,5113500256,Türk ve İslam Eserleri Müzesi,Turkish and Islamic Arts Museum,tourism:museum,museum,41.006280,28.974915,0.362063,62,12,0.85,0.35,0.995989,0.911765,0.834238,0,0,1,0,0,Turkish and Islamic Arts Museum,True,Turkish and Islamic Arts Museum,8174199.0,51912.0,35.531828,79.0,0.008086
8,3373094254,Halı Müzesi,Carpet Museum,tourism:museum,museum,41.005682,28.978669,0.280933,60,12,0.85,0.35,0.996972,0.882353,0.827180,0,0,1,0,0,Carpet Museum,True,Museum of Carpet,62201527.0,2016.0,1.598731,296.0,0.000364
9,527580309,Keçecizâde Fuad Paşa Türbesi,Turbe of Keçecizâde Fuad Pasha,historic:tomb,historic,41.006561,28.972843,0.500662,61,12,0.82,0.35,0.994309,0.897059,0.821057,0,1,0,0,1,Turbe of Keçecizâde Fuad Pasha,False,NaN,NaN,NaN,NaN,NaN,0.000000


## Combined importance
For now, we create a temporary combined score from:
- baseline heuristic importance
- Wikipedia popularity

In [185]:
top_pois_wiki["wiki_popularity_score"] = top_pois_wiki["wiki_popularity_score"].fillna(0)

top_pois_wiki["importance_with_wiki"] = (
    0.7 * top_pois_wiki["importance_score"]
    + 0.3 * top_pois_wiki["wiki_popularity_score"]
)

top_pois_wiki[[
    "name",
    "importance_score",
    "wiki_title",
    "wiki_popularity_score",
    "importance_with_wiki"
]].sort_values("importance_with_wiki", ascending=False).head(30)

,name,importance_score,wiki_title,wiki_popularity_score,importance_with_wiki
6,Ayasofya Tarih ve Deneyim Müzesi,0.834391,Hagia Sophia,1.000000,0.884073
2,İbrahim Paşa Sarayı,0.858915,Pargalı Ibrahim Pasha,0.129605,0.640122
17,Sultan III. Murad Türbesi,0.804154,Murad III,0.229397,0.631727
21,Sultan III. Mehmet Türbesi,0.800329,Mehmed III,0.222188,0.626887
5,Sultan Ahmet Türbesi,0.835898,Ahmed III,0.087470,0.611370
0,Lausos Sarayı'nın Kalıntıları,0.863004,NaN,0.000000,0.604103
1,Antiochos Sarayı'nın Kalıntıları,0.859224,NaN,0.000000,0.601457
3,Sağlık Müzesi,0.849310,NaN,0.000000,0.594517
4,Büyük Saray Mozaikleri Müzesi,0.844213,Great Palace Mosaic Museum,0.005564,0.592618
7,Türk ve İslam Eserleri Müzesi,0.834238,Turkish and Islamic Arts Museum,0.008086,0.586392


## Save intermediate output
This is still a first-pass file for the top POIs.
Later we can scale the matching logic to the full POI set.

In [186]:
top_pois_wiki.to_csv("../data/processed/poi_enriched_with_wiki_top200.csv", index=False)
print("Saved: ../data/processed/poi_enriched_with_wiki_top200.csv")
print("Shape:", top_pois_wiki.shape)

Saved: ../data/processed/poi_enriched_with_wiki_top200.csv
Shape: (50, 29)
